# Label Propagation — Training-Size Sensitivity

Reproduces APORIA-LP sensitivity to training size -- accuracy/F1 saturation as the labelled training set grows from 5 to 100 responses (Figures 4, 9).  
Complements also the results for the CoQA-89K dataset (Figure 11).

---

## Setup

In [ ]:
import os
import pathlib

# allow execution from either the repo root or code/
ROOT = pathlib.Path.cwd()
if ROOT.name == "code":
    os.chdir(ROOT.parent)

os.environ["OPENBLAS_NUM_THREADS"] = "4"


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.transforms import offset_copy

import aporia as ap


In [ ]:
# ---------------------------------------------------------------
# Load dataset + model metadata from a TOML config.
# ---------------------------------------------------------------
CONFIG_PATH = "config/socrates.toml"

cfg = ap.load_config(CONFIG_PATH)

# expose aliases used downstream
model_names           = cfg.model_names
figures_dir           = cfg.cache.fig_dir
model_latextags       = cfg.model_latextags
best_reg_lambda       = cfg.experiment.best_lambda
maxResponsesPerPrompt = cfg.dataset.max_responses_per_prompt


In [ ]:
plt.rcParams['text.usetex'] = True
plt.rcParams['text.latex.preamble'] = ap.matplotlib_latex_preamble(cfg)


## Data

In [ ]:
df = ap.load_dataframe(cfg)


In [ ]:
model_order, model_rank = ap.build_model_size_order(cfg)

false_premise = (
    {pid: fp for pid, fp in df[["prompt_id", "false_premise"]].value_counts().keys()}
    if "false_premise" in df.columns
    else None
)


## Experiment

In [ ]:
train_fractions = [x/100 for x in range(5, 101, 5)]   # 0.05 .. 1.00

results_lp = ap.run_full_label_propagation_study(
    df, cfg,
    projector_class  = ap.FisherProjection,
    projector_kwargs = {
        "lambda_reg":          best_reg_lambda,
        "normalise":           True,
        "normalise_by_trace":  True,
    },
    train_fractions  = train_fractions,
    n_iter           = 10,
    test_fraction    = 1/3,
    n_splits         = 5,
    ref_lambda_reg   = None,
    use_cache        = True,
    cache_dir        = f"{cfg.cache.root}/LP-trainingsize",
    overwrite_cache  = False,
    logskip          = True,
)


In [ ]:
agg_lc = ap.aggregate_metric_over_prompts(
    results_lp,
    metric="f1",
    agg_prompts=True,
    agg_models=False,
    agg_train_frac=False
)

## F1 vs. Training size

In [ ]:
def plot_learning_curves(
    agg_df,
    model_names,
    metric="f1",
    ratio=(3, 2),
    scale=3
):
    fig, ax = plt.subplots(figsize=[scale * x for x in ratio])

    for mid, name in model_names.items():
        df_m = agg_df[agg_df["model_id"] == mid]
        if df_m.empty:
            continue

        ax.plot(
            df_m["mean_n_train"],
            df_m["metric_mean"],
            marker="o",
            label=name
        )

        ax.fill_between(
            df_m["mean_n_train"],
            df_m["metric_mean"] - df_m["metric_std"],
            df_m["metric_mean"] + df_m["metric_std"],
            alpha=0.2
        )

    ax.set_xlabel("Training set size (mean)")
    ax.set_ylabel(metric.upper())
    ax.grid(True, alpha=0.4)
    ax.legend()
    ax.set_title(f"{metric.upper()} vs training size")

    return fig, ax

# ===== ===== ===== ===== ===== ===== ===== =====

fig, ax = plot_learning_curves(
    agg_lc,
    model_names=model_names,
)

## Effective train fraction plot

In [ ]:
results_lp.groupby('train_fraction')['n_train'].plot(color='k', alpha=.4);

## APORIA-LP Statistics -- Figure 8 (same as LabelPropation.ipynb)

In [ ]:
fig, _ = ap.plot_metric_boxplots_two_panels(
    results_lp,
    model_names=model_names,
    model_order=model_order,
    train_fraction=train_fractions[-1],
    width_ratios=[5, 9],
    xlims=[(0.5, 1.0), (.1, 1.0)],
    ratio=(3,1),
    scale=3
)

fig.savefig(f"{figures_dir}/LP_statistics.pdf", bbox_inches='tight')

## Heatmap statistics per model

In [ ]:
def plot_metric_heatmap(
    agg_df,
    model_names,
    metric="f1",
    ratio=(5, 3),
    scale=2
):
    
    def prepare_heatmap_df(agg_df):
        return agg_df.pivot(
            index="model_id",
            columns="train_fraction",
            values="metric_mean"
        )

    heat_df = prepare_heatmap_df(agg_df)
    
    fig, ax = plt.subplots(figsize=[scale * x for x in ratio])

    im = ax.imshow(
        heat_df.values,
        aspect="auto",
        origin="lower"
    )

    ax.set_yticks(range(len(heat_df.index)))
    ax.set_yticklabels([model_names[mid] for mid in heat_df.index])

    ax.set_xticks(range(len(heat_df.columns)))
    ax.set_xticklabels(
        [f"{int(100*c)}%" for c in heat_df.columns]
    )

    ax.set_xlabel("Training fraction")
    ax.set_title(metric.upper())

    fig.colorbar(im, ax=ax, label=metric.upper())

    return fig, ax

# ===== ===== ===== ===== ===== ===== ===== =====

plot_metric_heatmap(agg_lc, model_names)

## Constellation Plot for Accuracy and F1 -- Figures 4, 9, 11

In [ ]:
label_angles_f1 = {
    'Mistral-7B' : -150,
    'Gemma-9B' : -150,
    'Solar-11B' : -120,
    'Phi-14B' : 0,
    'Qwen-32B' : +30,
    'Gemma-27B' : -91,
    'Qwen-15B' : -45,
    'Llama-8B' : +45,
    'DeepSeek-7B': +30,
    'Llama2-13B' : 45,
    'Llama2-7B' : -30,
    'OPT-6.7B' : -160,
}

label_angles_acc = {
    'Mistral-7B' : -95,
    'Gemma-9B' : +60,
    'Solar-11B' : -165,
    'Phi-14B' : 0,
    'Qwen-32B' : 0,
    'Gemma-27B' : +20,
    'Llama-8B' : -95,
    'Qwen-15B' : -90,
    'Llama2-13B' : 45,
    'Llama2-7B' : -100,
    'OPT-6.7B' : 180,
}

In [ ]:
def plot_mean_vs_variability_single_metric(
    agg_df,
    model_names,
    metric_label,     # e.g. "metric" or "score"
    metric_name,      # e.g. "F1" or "Accuracy"
    label_angles=None,
    label_radius=10,
    ratio=(4, 6),
    scale=2,
    cmap_name=None,
):
    def alpha_from_fraction(tf, min_alpha=0.15, max_alpha=0.95):
        return min_alpha + (max_alpha - min_alpha) * tf

    def text_with_display_offset(ax, x, y, text, angle_deg, radius_pts,
                                color, fontsize=9):
        angle_rad = np.deg2rad(angle_deg)
        dx = radius_pts * np.cos(angle_rad)
        dy = radius_pts * np.sin(angle_rad)

        ha = "left" if -90 <= angle_deg <= 90 else "right"

        text_transform = offset_copy(
            ax.transData,
            fig=ax.figure,
            x=dx,
            y=dy,
            units="points"
        )

        ax.text(
            x, y, text,
            transform=text_transform,
            fontsize=fontsize,
            ha=ha,
            va="center",
            color=color,
        )

    fig, ax = plt.subplots(
        1, 1,
        figsize=[scale * x for x in ratio],
    )

    if cmap_name is not None:
        cmap = plt.get_cmap(cmap_name)
    else:
        cmap = ap.plotting.TAB_10

    for i, (mid, name) in enumerate(model_names.items()):
        df_m = (
            agg_df[agg_df["model_id"] == mid]
            .sort_values("mean_n_train")
        )
        if df_m.empty:
            continue

        color = cmap(i % cmap.N)

        x = df_m[f"{metric_label}_mean"].values
        y = df_m[f"{metric_label}_std"].values
        tfs = df_m["mean_n_train"].values
        tfs = tfs / tfs.max()

        # trajectory
        ax.plot(x, y, color=color, lw=1, alpha=0.35)

        # points
        for xi, yi, tf in zip(x, y, tfs):
            ax.scatter(
                xi, yi,
                color=color,
                s=35,
                alpha=alpha_from_fraction(tf),
                zorder=3,
            )

        # best point
        best_idx = np.argmax(x)
        ax.scatter(
            x[best_idx],
            y[best_idx],
            color=color,
            s=140,
            marker="*",
            edgecolor="black",
            linewidth=0.8,
            zorder=5,
        )

        # label
        if label_angles is not None:
            angle = label_angles.get(name, 0.0)
            text_with_display_offset(
                ax,
                x[best_idx],
                y[best_idx],
                name,
                angle_deg=angle,
                radius_pts=label_radius,
                color=color,
                fontsize=12,
            )

    ax.set_xlabel(f"{metric_name} mean performance across prompts", size=16)
    ax.set_ylabel(f"{metric_name} std across prompts", size=16)
    ax.grid(True, alpha=0.4)
    ax.tick_params(axis="both", labelsize=16)

    return fig, ax

# --- F1 plot
fig_f1, ax_f1 = plot_mean_vs_variability_single_metric(
    agg_lc,
    model_names,
    metric_label="metric",
    metric_name="F1",
    label_angles=label_angles_f1,
    ratio=(5, 3),
    scale=1.5,
    label_radius=12,
)

fig_f1.savefig(
    f"{figures_dir}/LP_performance_F1.pdf",
    bbox_inches="tight",
)


# --- Accuracy plot
fig_acc, ax_acc = plot_mean_vs_variability_single_metric(
    agg_lc,
    model_names,
    metric_label="score",
    metric_name="Accuracy",
    label_angles=label_angles_acc,
    ratio=(5, 3),
    scale=1.5,
    label_radius=12,
)

fig_acc.savefig(
    f"{figures_dir}/LP_performance_Accuracy.pdf",
    bbox_inches="tight",
)